# Entregável 1 — RecFair baseline (relatório)

> **Projeto:** RecFair — recomendação com contrato utilidade + justiça  
> **Arquitetura:** `baseline` · `prompt_version=v1`  
> **Runtime:** pacote `recfair/` · **Medição:** pacote `eval/`

Este notebook é **somente relatório**: especificação resumida e células que importam o sistema.

## Como reproduzir

```bash
cd recfair
python3.14 -m venv venv-recfair
source venv-recfair/bin/activate
cp .env.example .env   # GOOGLE_API_KEY
make install-dev
make kernel            # kernel Jupyter "Python (recfair)"
make chat ARCH=baseline
```

Golden-set: execute as células abaixo (`eval.runner.run_eval`).

## Estrutura herdada (import)

- Schema: `recfair.schemas.output.RecFairOutput`
- Golden-set: `data/golden/cases.json` + `eval.fingerprint.golden_revision()`
- Verify: `eval.verify.verify_case`
- Runner: `eval.runner.run_eval`

In [2]:
import json
import os

from IPython.display import HTML, display

from eval.fingerprint import golden_revision, load_cases
from eval.report import build_results_table, render_comparison_report, render_metrics_panel, summarize_records
from eval.runner import run_eval
from recfair.config import apply_dotenv, ensure_google_api_key, model_version
from recfair.observability.run_record import git_sha

apply_dotenv()
print("chave:", ensure_google_api_key())
cases = load_cases()
print(len(cases), "casos | golden_revision =", golden_revision(cases))
print("modelo:", model_version(), "| git:", git_sha())

chave: GOOGLE_API_KEY
30 casos | golden_revision = cedba68fb6c4c54c
modelo: gemini-3.5-flash-lite | git: 200da4bb64a0


## Execução do golden-set (baseline)

Reexecuta os 30 casos via `run_eval`. Requer `GOOGLE_API_KEY` no ambiente.

In [3]:
manifest = run_eval(arch="baseline", persist=True)
records = manifest["records"]
resumo = manifest["resumo"]
tabela = build_results_table(records)
display(HTML(render_comparison_report(tabela, "status_final")))
n_ok = int((tabela["status_final"] == "sucesso").sum())
n_err = int((tabela["status_final"] == "erro").sum())
n_err_star = int((tabela["status_final"] == "erro*").sum())
print(f"{len(records)} execuções · sucesso={n_ok} · erro={n_err} · erro*={n_err_star}")

Unexpected argument 'thinking_level' provided to ChatGoogleGenerativeAI. Did you mean: 'thinking_budget'?
/home/andersonbr/estudos/unicamp-llm-agents/recfair/recfair/graphs/baseline.py:147: UserWarning: WARNING! thinking_level is not default parameter.
                thinking_level was transferred to model_kwargs.
                Please confirm that thinking_level is what you intended.
  structured = _get_structured_llm()


caso,tipo caso,tipo teste,status final,baseline,gabarito,diff,motivo erro
T01,normal,restrito,sucesso,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → A8T3K5,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → A8T3K5,igual ao gabarito,—
T02,paráfrase,restrito,sucesso,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → A8T3K5,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → A8T3K5,igual ao gabarito,—
T03,composto,restrito,sucesso,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,F3P9W2 → L6K1C8 → 2Y8N4T → R5B7Q3 → 9C4M1H,igual ao gabarito,—
T04,normal,restrito,sucesso,24A51X → K8M2Q1 → 9P3W7C → B7F4L9 → Z5C1R8,24A51X → K8M2Q1 → 9P3W7C → B7F4L9 → Z5C1R8,igual ao gabarito,—
T05,janela_7d,restrito,erro,7K2N9A → Q4H8L2 → 3R1B6M → W9C5TD → 2M7K4F,7K2N9A → Q4H8L2 → 3R1B6M → 5J8P2X → 2M7K4F,extras na saída: ['W9C5TD']; faltam do gabarito: ['5J8P2X'],gabarito ≠ saída: extras na saída: ['W9C5TD']; faltam do gabarito: ['5J8P2X']
T06,informação ausente,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T07,informação ausente,restrito,sucesso,abstention · unknown_brand,abstention · unknown_brand,—,—
T08,ambíguo,restrito,sucesso,abstention · missing_category,abstention · missing_category,—,—
T09,ambíguo,restrito,erro,abstention · None,abstention · missing_category,—,"RF-04/05: reason=None, esperado missing_category"
T10,fora de escopo,restrito,erro,abstention · missing_category,abstention · unknown_category,—,"RF-04/05: reason=missing_category, esperado unknown_category"


30 execuções · sucesso=15 · erro=6 · erro*=9


## Métricas consolidadas

In [4]:
restrict_ok = resumo["e1_rate_restrict_n"]
restrict_n = resumo["e1_rate_restrict_d"]
overall_ok = resumo["e1_rate_overall_n"]
overall_n = resumo["e1_rate_overall_d"]
display(HTML(render_metrics_panel(resumo, restrict_ok, restrict_n, overall_ok, overall_n)))
print(json.dumps(resumo, indent=2, ensure_ascii=False))
if "run_path" in manifest:
    print("salvo:", manifest["run_path"])

e1_rate_restrictT01–T15 · baseline deve acertar,9/15 (60.0%)
e1_rate_overallT01–T30 · T16–T30 = gaps G,15/30 (50.0%)
Latência mediana,2.16 s
Latência média,2.05 s
Chamadas LLM,30
Tokens entrada / saída,"817,980 / 11,727"
Custo estimado (USD),$0.2747


{
  "e1_rate_restrict": 0.6,
  "e1_rate_restrict_n": 9,
  "e1_rate_restrict_d": 15,
  "e1_rate_overall": 0.5,
  "e1_rate_overall_n": 15,
  "e1_rate_overall_d": 30,
  "latencia_mediana_s": 2.16,
  "latencia_media_s": 2.05,
  "chamadas_llm": 30,
  "tokens_entrada": 817980,
  "tokens_saida": 11727,
  "custo_estimado_usd": 0.274711,
  "nota_custo": "$0.3/1M in, $2.5/1M out (thinking incl.)"
}
salvo: /home/andersonbr/estudos/unicamp-llm-agents/recfair/eval/runs/c7d7221e612b.json


## Análise crítica (E1)

O baseline stuffing exercita interpretação de NL sobre tabelas CSV, mas falha em desempates (`cod_sku`), diversidade de marca e abstenção em consultas ambíguas (ex.: T09). Gaps G (preço, claim, estoque, PII, injection) são **falhas previstas** no E1.

**Hipótese E2:** text-to-SQL / tools para agregação determinística da janela 7d e preço; workflow LangGraph com `recursion_limit` explícito.

**Pergunta obrigatória (curso):** comparar `e1_rate_restrict` e `e1_rate_overall` entre `ARCH=baseline` e incrementos futuros, mesmo modelo, baseline reexecutado na mesma sessão.